# Suite TYP — Types

`Types` renders schemas as Python dataclasses and reads dataclasses back as schemas, step by step: a session runs over
the schemas a store registers, Python's syntax trees and the output, which holds the module. `Dataclass` takes one
decision per object schema, whether its class is frozen; `Schema` reads one class back. Both round trips are tested:
schemas to source to schemas, and source to schemas to source.

In [ ]:
from mbse.Codegen.Python import Types
from mbse.Patterns import Transforms as T
from mbse.Programs.Python import Python312, Syntax as Py
from mbse.Schemas.Framework import JSON, Modules, Proxies, Schemas as S, Validators as V
from support import native, raises, stocked

B = Py.LANGUAGE.Builders
Phone = S.OfObject.Builder().name("Phone").properties(native("number")).create()
Contact = S.OfObject.Builder().name("Contact").ref().description('A "person"\nwith a home').properties(
    native("name"), native("age", int), lambda p: p.name("home").of(Phone),
    lambda p: p.name("tags").of(lambda t: t.as_indexed(lambda i: i.of(lambda x: x.as_native(str)))),
    lambda p: p.name("phones").of(lambda t: t.as_indexed(lambda i: i.of(Phone)))).create()
Empty = S.OfObject.Builder().name("Empty").create()


def module_of(schemas):
    """The schemas as a module's JSON text, to compare schemas by what they say."""
    store = Proxies.OfStore()
    return JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, schemas))


def registered(session, *names):
    """The schemas a reading session's store registers, by name."""
    schemas = session.store.stores[0].store
    return [schemas.registered(name) for name in names]


SOURCE = '''from __future__ import annotations
from dataclasses import dataclass


@dataclass(eq=False)
class Contact:
    "A \\"person\\"\\nwith a home"
    name: str | None = None
    age: int | None = None
    home: Phone | None = None
    tags: list[str] | None = None
    phones: list[Phone] | None = None


@dataclass
class Empty:
    pass


@dataclass(frozen=True)
class Phone:
    number: str | None = None
'''

## TYP-01 · The store: the schemas a store registers and refers to, Python's syntax trees, and the output holding the module

In [ ]:
module = B.Module().create()
store = Types.store(stocked(Contact, Empty), module)
assert list(store.extent("Schemas.Object")) == [Contact, Empty, Phone]  # Phone: referred to
output = store.singleton(Types.OUTPUT)
assert output.module is module and output.schema_name() == "Codegen.Output" and output.owner() is None
assert output.identity() == output.identity() and output.identity() != Types.Output().identity()
assert [e.targets["module"] for e in V.entries_of(output)["modules"]] == [module] and V.entries_of(Types.Output()) == {}
assert Types.Output.Schema.singleton == "Codegen.Output" and Types.Generated.links == ("output", "module")
assert store.extent("Programs.Python.Module") == (module,)  # reached from the output

## TYP-02 · Dataclass: a candidate per value of `frozen` for each object schema it renders and has no class for

In [ ]:
session = T.Session(Types.store(stocked(Contact, Empty), B.Module().create()), list(Types.TO_PYTHON))
assert [(c.match["s"].name, c.arguments) for c in session.candidates()] == [
    ("Contact", {"frozen": False}), ("Contact", {"frozen": True}), ("Empty", {"frozen": False}),
    ("Empty", {"frozen": True}), ("Phone", {"frozen": False}), ("Phone", {"frozen": True})]
assert Types.Dataclass.parameters["frozen"].description == "Whether the class is frozen"
Link = S.OfRelation.Builder().name("Link").links("a", "b").create()
Word = S.OfNative.Builder().name("Word").type(int).create()
Reach = S.OfUnion.Builder().name("Reach").branches(lambda b: b.name("phone").of(Phone)).create()


def nested(depth, item):
    """A list of lists, `depth` deep."""
    return item if depth == 0 else lambda t: t.as_indexed(lambda i: i.of(nested(depth - 1, item)))


def holding(name, spec, field="x"):
    return S.OfObject.Builder().name(name).properties(lambda p: p.name(field).of(spec)).create()


unrendered = [
    holding("Inline", lambda t: t.as_object(lambda o: o)), holding("Foreign", lambda t: t.as_native(
        lambda n: n.token("ccpp", "int32_t"))), holding("Bounded", lambda t: t.as_indexed(lambda i: i.of(Phone).extent(0, S.Form.Data("variable", {"name": "n"})))),
    holding("Union", lambda t: t.as_union(lambda u: u.branches(lambda b: b.name("phone").of(Phone)))), holding("Deep", nested(9, lambda t: t.as_native(lambda n: n.token("ccpp", "int32_t")))),
    S.OfObject.Builder().name("Generic").parameters(lambda q: q.name("n")).create()]
others = T.Session(Types.store(stocked(*unrendered), B.Module().create()), list(Types.TO_PYTHON))
assert {c.match["s"].name for c in others.candidates()} == {"Phone"}  # referred to; none of the others renders
assert len(Types.missing(others)) == len(unrendered) + 1 and others.run(Types.PLAIN) == 1 and others.done
assert [s.name for s in Types.missing(others)] == sorted(s.name for s in unrendered)  # completeness: what has no class
deep = Types.generate(stocked(holding("Deeper", nested(9, Phone))))  # lists nest to any depth
assert "    x: " + "list[" * 9 + "Phone" + "]" * 9 + " | None = None\n" in Types.text(deep) and Types.missing(deep) == []
named = Types.generate(stocked(holding("Named", Word)))  # a named native: an alias, which the field names
assert "type Word = int\n" in Types.text(named) and "    x: Word | None = None\n" in Types.text(named) and Types.missing(named) == []

## TYP-03 · Taking the steps: the caller's decisions and the policy's, the module printed, the trace

In [ ]:
session = T.Session(Types.store(stocked(Contact, Empty), B.Module().create()), list(Types.TO_PYTHON))
phone = next(c for c in session.candidates() if c.match["s"] is Phone and c.arguments["frozen"])
session.take(phone)  # the caller decides that Phone is frozen
assert session.run(Types.PLAIN) == 2 and session.done
assert Types.text(session) == SOURCE
assert [(step.transform, step.match, dict(step.arguments), step.by) for step in session.steps] == [
    ("Dataclass", (T.Link("s", "Phone"),), {"frozen": True}, "caller"),
    ("Dataclass", (T.Link("s", "Contact"),), {"frozen": False}, "policy"),
    ("Dataclass", (T.Link("s", "Empty"),), {"frozen": False}, "policy")]
# Each step links what it wrote, by role: a class; reading back, a schema. A trace names them by their paths.
assert [[(link.role, link.element.name.spelling) for link in step.wrote] for step in session.steps] == [
    [("class", "Phone")], [("class", "Contact")], [("class", "Empty")]]
assert session.wrote(session.store.singleton(Types.OUTPUT).module.body[4]) is session.steps[0]
traces = T.register(Proxies.OfStore())
assert ('"match": [{"symbol": "s", "element": "Phone"}], "wrote": [{"role": "class", "element": '
        '"Codegen.Output/modules[0]/children[4]"}]') in JSON.ToJSON(traces)(T.Trace, session.trace(traces))
back = Types.read(Python312.parse(SOURCE))
assert [(step.wrote[0].role, step.wrote[0].element.name) for step in back.steps] == [
    ("schema", "Contact"), ("schema", "Empty"), ("schema", "Phone")]
assert '"wrote": [{"role": "schema", "element": "Phone"}]' in JSON.ToJSON(traces)(T.Trace, back.trace(traces))
assert Types.text(Types.generate(stocked(Contact, Empty))) == SOURCE.replace("@dataclass(frozen=True)", "@dataclass")
frozen = Types.generate(stocked(Contact), T.Policy(T.Clause("Dataclass", {"frozen": True})))
assert "@dataclass(eq=False, frozen=True)\nclass Contact:" in Types.text(frozen)
assert Py.LANGUAGE.validate(session.store.singleton(Types.OUTPUT).module) == []  # a valid tree

## TYP-04 · Schema: a dataclass read back as a schema; names, references, lists, docstrings, `eq=False`; refusals

In [ ]:
reading = Types.read(Python312.parse(SOURCE))
contact, empty, phone = registered(reading, "Contact", "Empty", "Phone")
assert (contact.ref, contact.description, list(contact.properties)) == (
    True, 'A "person"\nwith a home', ["name", "age", "home", "tags", "phones"])
assert contact.properties["home"].type is phone and contact.properties["phones"].type.item is phone  # filled when read
assert contact.properties["age"].type == S.OfNative.Data(int) and contact.properties["tags"].type.item == S.OfNative.Data(str)
assert (empty.properties, phone.ref, list(phone.properties)) == ({}, False, ["number"])
assert [step.transform for step in reading.steps] == ["Schema"] * 3 and reading.done
written = '''@dataclass
class Plain:
    """A 'quoted' \\\\ text"""
    count: int
    note: str = "x"


class Ignored:
    x: int


@dataclass(frozen=True, eq=True)
class Frozen:
    f()
    x: int


@other
class Other:
    x: int
'''
target = stocked(Phone)
Types.read(Python312.parse(written), target)
assert target.registered("Plain").description == "A 'quoted' \\ text" and not target.registered("Frozen").ref
assert list(target.registered("Plain").properties) == ["count", "note"] and "Ignored" not in target.names()
assert target.registered("Frozen").description is None and "Other" not in target.names()
assert "Phone" in target.names()  # a store given is the one read into
assert Types.read(Python312.parse("@dataclass\nclass Single:\n    'single'\n")).done
assert registered(Types.read(Python312.parse("@dataclass\nclass Single:\n    'it\\'s'\n")), "Single")[0].description == "it's"
for annotation, shown in [("dict[str]", "dict[str]"), ("dict[str, int, bool]", "dict[str, int, bool]"),
                          ("set[int]", "set[int]"), ("typing.Any", "typing.Any"), ("x.y[int]", "x.y[int]"),
                          ("f().g[int]", "f().g[int]"), ("int | str", "int | str"), ("int | 3", "int | 3"), ("int + None", "int + None")]:
    with raises(ValueError, match=f"Bad.x: cannot read the annotation {shown}"):
        Types.read(Python312.parse(f"@dataclass\nclass Bad:\n    x: {annotation}\n"))

## TYP-05 · Round trips: schemas to source to the same schemas, and source to schemas to the same source

In [ ]:
back = registered(Types.read(Python312.parse(SOURCE)), "Contact", "Empty", "Phone")
assert module_of(back) == module_of([Contact, Empty, Phone])  # what the schemas say, frozen aside
session = T.Session(Types.store(stocked(*back), B.Module().create()), list(Types.TO_PYTHON))
session.take(next(c for c in session.candidates() if c.match["s"] is back[2] and c.arguments["frozen"]))
session.run(Types.PLAIN)
assert Types.text(session) == SOURCE  # the decision the trace keeps, taken again
assert Types.text(Types.read(Python312.parse(SOURCE))) == Python312.print(Python312.parse(SOURCE))  # read leaves the module

## TYP-06 · Regenerating after the schemas change: decisions taken again by schema name, a new schema asked, a gone one reported

In [ ]:
first = T.Session(Types.store(stocked(Contact, Empty), B.Module().create()), list(Types.TO_PYTHON))
first.take(next(c for c in first.candidates() if c.match["s"] is Phone and c.arguments["frozen"]))
first.run(Types.PLAIN)
Phone2 = S.OfObject.Builder().name("Phone").properties(native("number"), native("kind")).create()  # a field more
Address = S.OfObject.Builder().name("Address").properties(native("city")).create()  # new
Contact2 = S.OfObject.Builder().name("Contact").ref().properties(
    native("name"), lambda p: p.name("home").of(Phone2), lambda p: p.name("address").of(Address)).create()
again = T.Session(Types.store(stocked(Contact2), B.Module().create()), list(Types.TO_PYTHON), earlier=first.steps)
assert again.run() == 2 and [(step.key, step.by) for step in again.steps] == [
    ("Dataclass(s=Contact)", "reused"), ("Dataclass(s=Phone)", "reused")]
assert {c.match["s"].name for c in again.candidates()} == {"Address"}  # the one decision left to ask
assert [step.key for step in again.orphans] == ["Dataclass(s=Empty)"]  # Empty is gone
again.run(Types.PLAIN)
assert "@dataclass(frozen=True)\nclass Phone:\n    number: str | None = None\n    kind: str | None = None\n" in Types.text(again)
changes = T.diff(first.steps, again.steps)
assert ([s.key for s in changes.added], [s.key for s in changes.removed], changes.changed) == (
    ["Dataclass(s=Address)"], ["Dataclass(s=Empty)"], ())
assert Types.text(Types.generate(stocked(Contact2), earlier=first.steps)) == Types.text(again)

## TYP-07 · Dicts, nested lists and keyword fields, both ways

In [ ]:
SHEET = 'from __future__ import annotations\nfrom dataclasses import dataclass\n\n\n@dataclass\nclass Score:\n    value: float | None = None\n\n\n@dataclass\nclass Sheet:\n    from_: str | None = None\n    grid: list[list[int]] | None = None\n    scores: dict[str, Score] | None = None\n    tags: dict[int, list[bytes]] | None = None\n'
Score = S.OfObject.Builder().name("Score").properties(native("value", float)).create()
Sheet = S.OfObject.Builder().name("Sheet").properties(
    native("from"), lambda p: p.name("grid").of(lambda t: t.as_indexed(lambda i: i.of(lambda x: x.as_indexed(lambda j: j.of(
        lambda y: y.as_native(int)))))),
    lambda p: p.name("scores").of(lambda t: t.as_indexed(lambda i: i.key(lambda k: k.as_native(str)).of(Score))),
    lambda p: p.name("tags").of(lambda t: t.as_indexed(lambda i: i.key(lambda k: k.as_native(int)).of(
        lambda x: x.as_indexed(lambda j: j.of(lambda y: y.as_native(bytes))))))).create()
generated = Types.generate(stocked(Sheet))
assert Types.text(generated) == SHEET and Types.missing(generated) == []
back = registered(Types.read(Python312.parse(SHEET)), "Score", "Sheet")
assert module_of(back) == module_of([Score, Sheet]) and list(back[1].properties) == ["from", "grid", "scores", "tags"]
assert Types.text(Types.generate(stocked(*back))) == SHEET

## TYP-08 · Names Python cannot spell: written as the schemas have them, reported by path, and refused when the source is taken

In [ ]:
odd = Types.generate(stocked(holding("Bad-Name", Phone), holding("class", Phone), holding("Dashed", Phone, "first-name")))
assert odd.done and Types.missing(odd) == []  # every schema has its class
PROBLEMS = ["body[2].name: 'Bad-Name' is not an identifier", "body[3].body[0].target.id: 'first-name' is not an identifier",
            "body[5].name: 'class' is a keyword"]
assert Types.problems(odd) == PROBLEMS  # the paths of the names
with raises(ValueError, match="the module is not valid Python: " + "; ".join(PROBLEMS)):
    Types.text(odd)
assert Types.problems(Types.generate(stocked(Contact))) == []

## TYP-09 · Relations: an entry class per relation, a field per adjacency holding its entries, read alike from proxies and generated classes

In [ ]:
ENTRIES = 'from __future__ import annotations\nfrom dataclasses import dataclass, field\nfrom typing import ClassVar\n\n\n@dataclass(eq=False)\nclass Contact:\n    name: str | None = None\n    phones: tuple[Phones, ...] = ()\n\n\n@dataclass(eq=False)\nclass Pager:\n    owners: tuple[Phones, ...] = ()\n\n\n@dataclass(eq=False)\nclass Parentage:\n    LINKS: ClassVar[tuple[str, ...]] = ("parent", "child")\n    parent: Person | None = None\n    child: Person | None = None\n\n\n@dataclass(eq=False)\nclass Person:\n    children: tuple[Parentage, ...] = field(default=(), metadata={"me": "parent"})\n    parents: tuple[Parentage, ...] = field(default=(), metadata={"me": "child"})\n\n\n@dataclass(eq=False)\nclass Phone:\n    number: str | None = None\n    owners: tuple[Phones, ...] = ()\n\n\n@dataclass(eq=False)\nclass Phones:\n    "A number to call"\n    LINKS: ClassVar[tuple[str, ...]] = ("owner", "phone")\n    UNIQUES: ClassVar[tuple[tuple[str, ...], ...]] = (("label", "owner"), ("phone",))\n    owner: Contact | None = None\n    phone: Pager | Phone | None = None\n    label: str | None = None\n'
Phones = S.OfRelation.Builder().name("Phones").description("A number to call").links("owner", "phone").properties(
    native("label")).unique("owner", "label").unique("phone").create()
Parentage = S.OfRelation.Builder().name("Parentage").links("parent", "child").create()
Card = S.OfObject.Builder().name("Contact").ref().properties(native("name")).relations(
    lambda r: r.name("phones").of(Phones).me("owner")).create()
Mobile = S.OfObject.Builder().name("Phone").ref().properties(native("number")).relations(
    lambda r: r.name("owners").of(Phones).me("phone")).create()
Pager = S.OfObject.Builder().name("Pager").ref().relations(lambda r: r.name("owners").of(Phones).me("phone")).create()
Person = S.OfObject.Builder().name("Person").ref().relations(lambda r: r.name("children").of(Parentage).me("parent"),
                                                             lambda r: r.name("parents").of(Parentage).me("child")).create()
MODEL = [Card, Pager, Person, Mobile, Parentage, Phones]
related = Types.generate(stocked(*MODEL))
assert Types.text(related) == ENTRIES and Types.missing(related) == []
assert [step.key for step in related.steps][-2:] == ["Entry(r=Parentage)", "Entry(r=Phones)"]  # a step per entry class
back = registered(Types.read(Python312.parse(ENTRIES)), *(schema.name for schema in MODEL))
assert module_of(back) == module_of(MODEL)  # links, properties, uniques, descriptions and adjacencies, as they were
assert Types.text(Types.generate(stocked(*back))) == ENTRIES


def numbers(contact):
    """Code that reads an adjacency's entries: the same for proxies and generated classes."""
    return sorted((entry.label, entry.phone.number) for entry in contact.phones)


proxies = stocked(*MODEL)
ann = proxies.Contact().name("ann").phones(lambda e: e.phone(proxies.Phone().number("1").create()).label("home")).create()
generated = {}
exec(ENTRIES, generated)
twin, phone = generated["Contact"](name="ann"), generated["Phone"](number="1")
entry = generated["Phones"](owner=twin, phone=phone, label="home")
twin.phones, phone.owners = (entry,), (entry,)
assert numbers(ann) == numbers(twin) == [("home", "1")] and ann.phones[0].owner is ann and twin.phones[0].owner is twin
# What renders no entry class, and what reads back no relation.
Lone = S.OfRelation.Builder().name("Lone").links("a", "b").create()  # nobody declares b
Generic = S.OfRelation.Builder().name("Generic").links("a", "b").parameters(lambda q: q.name("n")).create()
Reaching = S.OfRelation.Builder().name("Reaching").links("a", "b").properties(lambda q: q.name("x").of(
    S.OfUnion.Builder().branches(lambda b: b.name("n").of(lambda t: t.as_native(int))).create())).create()  # inline
ends = lambda relation: [S.OfObject.Builder().name(f"{relation.name}{link.upper()}").ref().relations(  # noqa: E731
    lambda q, link=link: q.name("xs").of(relation).me(link)).create() for link in relation.links]
assert [s.name for s in Types.missing(Types.generate(stocked(*ends(Lone)[:1])))] == ["Lone"]
assert [s.name for s in Types.missing(Types.generate(stocked(*ends(Generic), *ends(Reaching))))] == ["Generic", "Reaching"]
Unnamed = S.OfRelation.Builder().links("a", "b").create()
assert [s.name for s in Types.missing(Types.generate(stocked(S.OfObject.Builder().name("Holder").relations(
    lambda q: q.name("xs").of(Unnamed).me("a")).create())))] == ["Holder"]  # an unnamed relation has no entry class
with raises(ValueError, match="W.xs: Unknown is not a relation the module or the store holds"):
    Types.read(Python312.parse("@dataclass\nclass W:\n    xs: tuple[Unknown, ...] = ()\n"))
with raises(ValueError, match="Person.children: cannot tell which link of Parentage it is from"):
    Types.read(Python312.parse(ENTRIES.replace('field(default=(), metadata={"me": "parent"})', "()")))
held = stocked(Phones, Card, Mobile)  # a relation the store holds, with no class in the module, so its link is given
Types.read(Python312.parse('@dataclass\nclass Pager:\n    owners: tuple[Phones, ...] = field(default=(), metadata={"me": "phone"})\n'),
           held)
assert held.registered("Pager").adjacencies["owners"].relation is Phones and held.registered("Pager").adjacencies["owners"].me == "phone"
with raises(ValueError, match="Pager.owners: cannot tell which link of Phones it is from"):
    Types.read(Python312.parse("@dataclass\nclass Pager:\n    owners: tuple[Phones, ...] = ()\n"), stocked(Phones, Card, Mobile))
odd = Types.read(Python312.parse(  # a link of another annotation types nothing; the link typed by W is the one
    '@dataclass(eq=False)\nclass R:\n    LINKS: ClassVar[tuple[str, ...]] = ("a", "b")\n    a: list[W] | None = None\n'
    '    b: W | None = None\n\n\n@dataclass\nclass W:\n    xs: tuple[R, ...] = ()\n'))
assert registered(odd, "W")[0].adjacencies["xs"].me == "b"

## TYP-10 · Unions and intersections: a class of a field per branch or part, read alike from proxies and generated classes

In [ ]:
VARIANTS = 'from __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import ClassVar\n\n\n@dataclass(eq=False)\nclass Card:\n    reach: Reach | None = None\n    stamps: list[Stamp] | None = None\n\n\n@dataclass\nclass Email:\n    address: str | None = None\n\n\n@dataclass\nclass Phone:\n    number: str | None = None\n\n\n@dataclass\nclass Reach:\n    "How to reach"\n    KIND: ClassVar[str] = "union"\n    phone: Phone | None = None\n    email: Email | None = None\n    from_: Phone | None = None\n\n\n@dataclass\nclass Stamp:\n    KIND: ClassVar[str] = "intersection"\n    when: When | None = None\n    who: Who | None = None\n\n\n@dataclass\nclass When:\n    at: int | None = None\n\n\n@dataclass\nclass Who:\n    by: str | None = None\n'
Email = S.OfObject.Builder().name("Email").properties(native("address")).create()
Reach = S.OfUnion.Builder().name("Reach").description("How to reach").branches(
    lambda b: b.name("phone").of(Phone), lambda b: b.name("email").of(Email), lambda b: b.name("from").of(Phone)).create()
When = S.OfObject.Builder().name("When").properties(native("at", int)).create()
Who = S.OfObject.Builder().name("Who").properties(native("by")).create()
Stamp = S.OfIntersection.Builder().name("Stamp").parts(lambda b: b.name("when").of(When), lambda b: b.name("who").of(Who)).create()
Badge = S.OfObject.Builder().name("Card").ref().properties(lambda p: p.name("reach").of(Reach), lambda p: p.name("stamps").of(
    lambda x: x.as_indexed(lambda i: i.of(Stamp)))).create()
assert [schema.validate() for schema in (Reach, Stamp, Badge)] == [[], [], []]
varied = Types.generate(stocked(Badge))
assert Types.text(varied) == VARIANTS and Types.missing(varied) == []
assert [step.key for step in varied.steps][-2:] == ["Union(s=Reach)", "Intersection(s=Stamp)"]
back = registered(Types.read(Python312.parse(VARIANTS)), "Card", "Email", "Phone", "Reach", "Stamp", "When", "Who")
assert module_of(back) == module_of([Badge, Email, Phone, Reach, Stamp, When, Who])  # branches, parts, a keyword's, a description
assert Types.text(Types.generate(stocked(*back))) == VARIANTS


def reached(card):
    """Code that reads a union value's branches: the same for proxies and generated classes."""
    return card.reach.email.address if card.reach.phone is None else card.reach.phone.number


proxies = stocked(Badge)
generated = {}
exec(VARIANTS, generated)
assert reached(proxies.Card().reach(lambda u: u.email(lambda e: e.address("a@x"))).create()) == reached(
    generated["Card"](reach=generated["Reach"](email=generated["Email"](address="a@x")))) == "a@x"
frozen = Types.generate(stocked(Badge), T.Policy(T.Clause("Dataclass", {"frozen": False}), T.Clause("Union", {"frozen": True}),
                                                 T.Clause("Intersection", {"frozen": True})))
assert "@dataclass(frozen=True)\nclass Reach:" in Types.text(frozen) and "@dataclass(frozen=True)\nclass Stamp:" in Types.text(frozen)
# What renders no class: no branches, parameters, a branch of an inline type.
Empty = S.OfUnion.Builder().name("Empty").create()
Generic = S.OfIntersection.Builder().name("Generic").parameters(lambda q: q.name("n")).parts(
    lambda b: b.name("a").of(lambda x: x.as_native(int))).create()
Inline = S.OfUnion.Builder().name("Inline").branches(lambda b: b.name("a").of(lambda x: x.as_object(lambda o: o))).create()
holder = S.OfObject.Builder().name("Holder").properties(*[lambda p, t=t: p.name(t.name.lower()).of(t) for t in (Empty, Generic, Inline)]).create()
assert [s.name for s in Types.missing(Types.generate(stocked(holder)))] == ["Empty", "Inline", "Generic"]
with raises(ValueError, match="W.x: Elsewhere is not a class of the module or a schema of the store"):
    Types.read(Python312.parse("@dataclass\nclass W:\n    x: Elsewhere | None = None\n"))  # reading never makes one up

## TYP-11 · Flat unions and intersections: a type alias of the branches' types, and a class of the parts' properties, as proxies read them

In [ ]:
FLAT = 'from __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Annotated, ClassVar\n\n\n@dataclass\nclass Call:\n    number: str | None = None\n\n\ntype Channel = Call | Mail\ntype Code = Annotated[int | str, {"branches": ["n", "s"], "description": "A code"}]\n\n\n@dataclass\nclass Mail:\n    address: str | None = None\n\n\n@dataclass\nclass Stamped:\n    KIND: ClassVar[str] = "intersection"\n    PARTS: ClassVar[dict[str, str | tuple[str, ...]]] = {"when": ("at",), "who": "Who"}\n    at: int | None = None\n    by: str | None = None\n\n\n@dataclass(eq=False)\nclass Ticket:\n    channel: Channel | None = None\n    code: Code | None = None\n    stamp: Stamped | None = None\n\n\n@dataclass\nclass Who:\n    by: str | None = None\n'
Mail = S.OfObject.Builder().name("Mail").properties(native("address")).create()
Call = S.OfObject.Builder().name("Call").properties(native("number")).create()
Channel = S.OfUnion.Builder().name("Channel").branches(lambda b: b.name("call").of(Call), lambda b: b.name("mail").of(Mail)).flat().create()
Code = S.OfUnion.Builder().name("Code").description("A code").branches(
    lambda b: b.name("n").of(lambda x: x.as_native(int)), lambda b: b.name("s").of(lambda x: x.as_native(str))).flat().create()
Stamped = S.OfIntersection.Builder().name("Stamped").parts(
    lambda b: b.name("when").of(lambda x: x.as_object(lambda o: o.properties(native("at", int)))),
    lambda b: b.name("who").of(Who)).flat().create()
Ticket = S.OfObject.Builder().name("Ticket").ref().properties(
    lambda p: p.name("channel").of(Channel), lambda p: p.name("code").of(Code), lambda p: p.name("stamp").of(Stamped)).create()
assert [schema.validate() for schema in (Channel, Code, Stamped, Ticket)] == [[], [], [], []]
flat = Types.generate(stocked(Ticket))
assert Types.text(flat) == FLAT and Types.missing(flat) == []
assert [step.key for step in flat.steps][-3:] == ["Intersection(s=Stamped)", "Alias(s=Channel)", "Alias(s=Code)"]
assert [link.role for step in flat.steps[-3:] for link in step.wrote] == ["class", "alias", "alias"]
back = registered(Types.read(Python312.parse(FLAT)), "Call", "Channel", "Code", "Mail", "Stamped", "Ticket", "Who")
assert module_of(back) == module_of([Call, Channel, Code, Mail, Stamped, Ticket, Who])  # flat, names, an inline part, a description
assert Types.text(Types.generate(stocked(*back))) == FLAT


def read(ticket):
    """Code that reads flat values: the same for proxies and generated classes."""
    channel = ticket.channel.address if hasattr(ticket.channel, "address") else ticket.channel.number
    return channel, ticket.code, ticket.stamp.at, ticket.stamp.by


import sys, types
module = types.ModuleType("flat_generated")  # a module of its own, as dataclasses needs for its ClassVars
sys.modules[module.__name__] = module
exec(FLAT, module.__dict__)
proxies = stocked(Ticket)
mail = proxies.Ticket().channel(lambda u: u.mail(lambda m: m.address("a@x"))).create().channel
proxy = proxies.Ticket().channel(mail).code(7).stamp(lambda s: s.at(3).by("me")).create()
twin = module.Ticket(channel=module.Mail(address="a@x"), code=7, stamp=module.Stamped(at=3, by="me"))
assert read(proxy) == read(twin) == ("a@x", 7, 3, "me")
# Branches named as their types are, and no description: a plain alias, read back as such.
plain = S.OfUnion.Builder().name("Plain").branches(lambda b: b.name("int").of(lambda x: x.as_native(int)),
    lambda b: b.name("list").of(lambda x: x.as_indexed(lambda i: i.of(lambda y: y.as_native(str)))),
    lambda b: b.name("dict").of(lambda x: x.as_indexed(lambda i: i.key(lambda k: k.as_native(str)).of(lambda y: y.as_native(int))))).flat().create()
holding = S.OfObject.Builder().name("Holding").properties(lambda p: p.name("x").of(plain)).create()
assert "type Plain = int | list[str] | dict[str, int]\n" in Types.text(Types.generate(stocked(holding)))
assert module_of(registered(Types.read(Python312.parse(Types.text(Types.generate(stocked(holding))))), "Plain")) == module_of([plain])
# What renders no alias or class: a flat intersection's part whose properties do not render, or a native part.
odd = S.OfIntersection.Builder().name("Odd").parts(lambda b: b.name("a").of(lambda x: x.as_object(lambda o: o.properties(
    lambda q: q.name("w").of(lambda y: y.as_native(lambda n: n.token("ccpp", "x"))))))).flat().create()
named = S.OfObject.Builder().name("Named").properties(lambda q: q.name("w").of(lambda y: y.as_native(lambda n: n.token("ccpp", "x")))).create()
odder = S.OfIntersection.Builder().name("Odder").parts(lambda b: b.name("a").of(named)).flat().create()
held = S.OfObject.Builder().name("Held").properties(lambda p: p.name("o").of(odd), lambda p: p.name("p").of(odder)).create()
assert [s.name for s in Types.missing(Types.generate(stocked(held)))] == ["Named", "Odd", "Odder"]

## TYP-12 · Nothing is lost: a singleton and every description written and read back, and what Python's types cannot hold left without a class

In [ ]:
KEPT = 'from __future__ import annotations\nfrom dataclasses import dataclass, field\nfrom typing import Annotated, ClassVar\n\n\n@dataclass\nclass By:\n    who: str | None = None\n\n\ntype Code = Annotated[int | str, {"branches": ["n", "s"], "descriptions": {"n": "a number"}}]\n\n\n@dataclass(eq=False)\nclass Labels:\n    LINKS: ClassVar[tuple[str, ...]] = ("owner", "phone")\n    owner: Owner | None = None\n    phone: Pager | None = None\n    label: str | None = field(default=None, metadata={"description": "home or work"})\n\n\n@dataclass(eq=False)\nclass Owner:\n    SINGLETON: ClassVar[str] = "The.Owner"\n    via: Via | None = None\n    code: Code | None = None\n    seal: Seal | None = None\n    pagers: tuple[Labels, ...] = field(default=(), metadata={"description": "the pagers it owns"})\n\n\n@dataclass\nclass Pager:\n    number: str | None = field(default=None, metadata={"description": "digits"})\n    owners: tuple[Labels, ...] = ()\n\n\n@dataclass\nclass Seal:\n    KIND: ClassVar[str] = "intersection"\n    PARTS: ClassVar[dict[str, str | tuple[str, ...]]] = {"when": ("at",), "by": "By"}\n    DESCRIPTIONS: ClassVar[dict[str, str]] = {"when": "the time"}\n    at: int | None = field(default=None, metadata={"description": "seconds"})\n    who: str | None = None\n\n\n@dataclass\nclass Via:\n    KIND: ClassVar[str] = "union"\n    pager: Pager | None = field(default=None, metadata={"description": "by pager"})\n    by: By | None = None\n'
Labels = S.OfRelation.Builder().name("Labels").links("owner", "phone").properties(
    lambda p: p.name("label").description("home or work").of(lambda x: x.as_native(str))).create()
Pager = S.OfObject.Builder().name("Pager").properties(lambda p: p.name("number").description("digits").of(
    lambda x: x.as_native(str))).relations(lambda r: r.name("owners").of(Labels).me("phone")).create()
By = S.OfObject.Builder().name("By").properties(native("who")).create()
Via = S.OfUnion.Builder().name("Via").branches(lambda b: b.name("pager").description("by pager").of(Pager),
                                               lambda b: b.name("by").of(By)).create()
Code = S.OfUnion.Builder().name("Code").branches(lambda b: b.name("n").description("a number").of(lambda x: x.as_native(int)),
                                                 lambda b: b.name("s").of(lambda x: x.as_native(str))).flat().create()
Seal = S.OfIntersection.Builder().name("Seal").parts(
    lambda b: b.name("when").description("the time").of(lambda x: x.as_object(lambda o: o.properties(
        lambda p: p.name("at").description("seconds").of(lambda y: y.as_native(int))))),
    lambda b: b.name("by").of(By)).flat().create()
Owner = S.OfObject.Builder().name("Owner").singleton("The.Owner").properties(
    lambda p: p.name("via").of(Via), lambda p: p.name("code").of(Code), lambda p: p.name("seal").of(Seal)).relations(
    lambda r: r.name("pagers").description("the pagers it owns").of(Labels).me("owner")).create()
assert [schema.validate() for schema in (Labels, Pager, Via, Code, Seal, Owner)] == [[]] * 6
kept = Types.generate(stocked(Pager, Owner))
assert Types.text(kept) == KEPT and Types.missing(kept) == []
back = registered(Types.read(Python312.parse(KEPT)), "By", "Code", "Labels", "Owner", "Pager", "Seal", "Via")
assert module_of(back) == module_of([By, Code, Labels, Owner, Pager, Seal, Via])  # every description, and the singleton
assert Types.text(Types.generate(stocked(*back))) == KEPT
plain, = registered(Types.read(Python312.parse("@dataclass\nclass Plain:\n    x: int | None = field(default=None)\n")), "Plain")
assert plain.properties["x"].description is None  # a field without metadata
# What waits on parameters' Python form has no class, rather than one that loses it: a width that is a term, parameters.
held = [S.OfObject.Builder().name(name).properties(lambda p, spec=spec: p.name("x").of(lambda t: t.as_native(spec))).create()
        for name, spec in (("Sized", lambda n: n.type(int).bits(S.Form.Data("variable", {"name": "n"}))),
                           ("Typed", lambda n: n.type(int).parameters(lambda q: q.name("w"))))]
assert [s.name for s in Types.missing(Types.generate(stocked(*held)))] == ["Sized", "Typed"]

## TYP-13 · Named natives and lists: a type alias of what each holds, which a field names, read back

In [ ]:
NAMED = 'from __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Annotated\ntype Count = int\n\n\n@dataclass\nclass Ledger:\n    word: Word | None = None\n    names: Names | None = None\n    phones: Phones | None = None\n    tally: Tally | None = None\n    mark: Mark | None = None\n\n\ntype Mark = Word | Count\ntype Names = list[Word]\n\n\n@dataclass\nclass Phone:\n    number: str | None = None\n\n\ntype Phones = list[Phone]\ntype Tally = Annotated[dict[Word, Count], {"description": "counts by word"}]\ntype Word = Annotated[str, {"description": "a word"}]\n'
Word = S.OfNative.Builder().name("Word").type(str).description("a word").create()
Count = S.OfNative.Builder().name("Count").type(int).create()
Names = S.OfIndexed.Builder().name("Names").of(Word).create()
Phones = S.OfIndexed.Builder().name("Phones").of(Phone).create()
Tally = S.OfIndexed.Builder().name("Tally").key(Word).of(Count).description("counts by word").create()
Mark = S.OfUnion.Builder().name("Mark").branches(lambda b: b.name("word").of(Word), lambda b: b.name("count").of(Count)).flat().create()
Ledger = S.OfObject.Builder().name("Ledger").properties(*[lambda p, t=t: p.name(t.name.lower()).of(t) for t in (Word, Names, Phones, Tally, Mark)]).create()
assert [schema.validate() for schema in (Word, Count, Names, Phones, Tally, Mark, Ledger)] == [[]] * 7
named = Types.generate(stocked(Ledger))
assert Types.text(named) == NAMED and Types.missing(named) == []
assert [link.role for step in named.steps if step.transform.endswith("Alias") for link in step.wrote] == ["alias"] * 6
back = registered(Types.read(Python312.parse(NAMED)), "Count", "Ledger", "Mark", "Names", "Phone", "Phones", "Tally", "Word")
assert module_of(back) == module_of([Count, Ledger, Mark, Names, Phone, Phones, Tally, Word])  # natives, lists, a description
assert Types.text(Types.generate(stocked(*back))) == NAMED
alone = Types.read(Python312.parse("type Word = str\n"))  # an alias no class names: read by its own step
assert [(link.role, link.element.name, link.element.token.name) for link in alone.steps[0].wrote] == [("schema", "Word", "str")]
# What has no alias: another format's native, a list bounded by a term, a list of an inline object; a field names each anyway.
Wide = S.OfNative.Builder().name("Wide").token("ccpp", "int32_t").create()
Bounded = S.OfIndexed.Builder().name("Bounded").of(Phone).extent(0, S.Form.Data("variable", {"name": "n"})).create()
Odd = S.OfIndexed.Builder().name("Odd").of(lambda x: x.as_object(lambda o: o)).create()
uses = S.OfObject.Builder().name("Uses").properties(*[lambda p, t=t: p.name(t.name.lower()).of(t) for t in (Wide, Bounded, Odd)]).create()
assert [s.name for s in Types.missing(Types.generate(stocked(uses)))] == ["Wide", "Bounded", "Odd"]
# Reading refuses an alias of what no schema is, and a list that holds itself through aliases alone.
with raises(ValueError, match="X: cannot read the alias of Phone"):
    Types.read(Python312.parse("type X = Phone\n"))
with raises(ValueError, match="A: a list that holds itself through aliases alone has no Python form"):
    Types.read(Python312.parse("type A = list[A]\n"))

## TYP-14 · Widths, extents and natives' own descriptions: `Annotated` metadata, read back

In [ ]:
FACETED = 'from __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Annotated\ntype Byte = Annotated[int, {"bits": 8, "description": "one octet"}]\ntype Few = Annotated[list[Phone], {"minimum": 1, "maximum": 3}]\n\n\n@dataclass\nclass Meter:\n    reading: Annotated[int, {"bytes": 4}] | None = None\n    count: Annotated[int, {"description": "a count"}] | None = None\n    last: Annotated[list[float], {"minimum": 0}] | None = None\n    raw: Annotated[str, {"native": ["python3", "str"]}] | None = None\n    byte: Byte | None = None\n    tick: Tick | None = None\n    few: Few | None = None\n    size: Size | None = None\n\n\n@dataclass\nclass Phone:\n    number: str | None = None\n\n\ntype Size = Annotated[int, {"bits": 16}] | str\ntype Tick = Annotated[int, {"native": ["python3", "int"]}]\n'
Byte = S.OfNative.Builder().name("Byte").type(int).bits(8).description("one octet").create()
Tick = S.OfNative.Builder().name("Tick").token("python3", "int").create()
Few = S.OfIndexed.Builder().name("Few").of(Phone).extent(1, 3).create()
Size = S.OfUnion.Builder().name("Size").branches(lambda b: b.name("int").of(lambda x: x.as_native(lambda n: n.type(int).bits(16))),
                                                 lambda b: b.name("str").of(lambda x: x.as_native(str))).flat().create()
Meter = S.OfObject.Builder().name("Meter").properties(
    lambda p: p.name("reading").of(lambda t: t.as_native(lambda n: n.type(int).bytes(4))),
    lambda p: p.name("count").of(lambda t: t.as_native(lambda n: n.type(int).description("a count"))),
    lambda p: p.name("last").of(lambda t: t.as_indexed(lambda i: i.of(lambda y: y.as_native(float)).extent(0))),
    lambda p: p.name("raw").of(lambda t: t.as_native(lambda n: n.token("python3", "str"))),
    lambda p: p.name("byte").of(Byte), lambda p: p.name("tick").of(Tick), lambda p: p.name("few").of(Few),
    lambda p: p.name("size").of(Size)).create()
assert [schema.validate() for schema in (Byte, Tick, Few, Size, Meter)] == [[]] * 5
faceted = Types.generate(stocked(Meter))
assert Types.text(faceted) == FACETED and Types.missing(faceted) == []
back = registered(Types.read(Python312.parse(FACETED)), "Byte", "Few", "Meter", "Phone", "Size", "Tick")
assert module_of(back) == module_of([Byte, Few, Meter, Phone, Size, Tick])  # widths, extents, descriptions, tokens; a branch named as its type
assert Types.text(Types.generate(stocked(*back))) == FACETED
Exact = S.OfNative.Builder().name("Exact").token("python3", "decimal.Decimal").create()  # no host type in proxies: no alias
assert [s.name for s in Types.missing(Types.generate(stocked(S.OfObject.Builder().name("Money").properties(
    lambda p: p.name("x").of(Exact)).create())))] == ["Exact"]

## TYP-15 · Keyed lists that a `dict` cannot hold faithfully: `Proxies.OfIndexed.Map`, read alike from proxies and generated classes

In [ ]:
KEYED = 'from __future__ import annotations\nfrom dataclasses import dataclass\nfrom mbse.Schemas.Framework import Proxies\ntype Labels = Proxies.OfIndexed.Map[Point, str]\n\n\n@dataclass\nclass Point:\n    x: float | None = None\n    y: float | None = None\n\n\ntype Shape = list[int] | Proxies.OfIndexed.Map[float, int]\n\n\n@dataclass(eq=False)\nclass Survey:\n    weights: Proxies.OfIndexed.Map[float, int] | None = None\n    paths: Proxies.OfIndexed.Map[list[int], str] | None = None\n    names: dict[str, int] | None = None\n    labels: Labels | None = None\n    shape: Shape | None = None\n'
Point = S.OfObject.Builder().name("Point").properties(native("x", float), native("y", float)).create()
Labels = S.OfIndexed.Builder().name("Labels").key(Point).of(lambda t: t.as_native(str)).create()
Shape = S.OfUnion.Builder().name("Shape").branches(
    lambda b: b.name("list").of(lambda t: t.as_indexed(lambda i: i.of(lambda v: v.as_native(int)))),
    lambda b: b.name("dict").of(lambda t: t.as_indexed(lambda i: i.key(lambda k: k.as_native(float)).of(lambda v: v.as_native(int))))).flat().create()
Survey = S.OfObject.Builder().name("Survey").ref().properties(
    lambda p: p.name("weights").of(lambda t: t.as_indexed(lambda i: i.key(lambda k: k.as_native(float)).of(lambda v: v.as_native(int)))),
    lambda p: p.name("paths").of(lambda t: t.as_indexed(lambda i: i.key(lambda k: k.as_indexed(lambda j: j.of(lambda y: y.as_native(int)))).of(
        lambda v: v.as_native(str)))),
    lambda p: p.name("names").of(lambda t: t.as_indexed(lambda i: i.key(lambda k: k.as_native(str)).of(lambda v: v.as_native(int)))),
    lambda p: p.name("labels").of(Labels), lambda p: p.name("shape").of(Shape)).create()
assert [schema.validate() for schema in (Point, Labels, Shape, Survey)] == [[]] * 4
keyed = Types.generate(stocked(Survey))
assert Types.text(keyed) == KEYED and Types.missing(keyed) == []
back = registered(Types.read(Python312.parse(KEYED)), "Labels", "Point", "Shape", "Survey")
assert module_of(back) == module_of([Labels, Point, Shape, Survey])  # keys: a float, a list, a value object; a flat union of lists
assert Types.text(Types.generate(stocked(*back))) == KEYED


def weighed(survey):
    """Code that reads keyed lists: the same for proxies and generated classes, keys compared as schema equality does."""
    return survey.weights[float("nan")], survey.weights[-0.0], survey.weights.get(0.0), survey.paths[[1, 2]]


import sys, types
keyed_module = types.ModuleType("keyed_generated")  # a module of its own, as dataclasses needs
sys.modules[keyed_module.__name__] = keyed_module
exec(KEYED, keyed_module.__dict__)
pairs = [(float("nan"), 1), (-0.0, 2)]
proxy = stocked(Survey).Survey().weights(pairs).paths([((1, 2), "a")]).create()
twin = keyed_module.Survey(weights=Proxies.OfIndexed.Map(pairs), paths=Proxies.OfIndexed.Map([((1, 2), "a")]))
assert weighed(proxy) == weighed(twin) == (1, 2, None, "a")
labels = Proxies.OfIndexed.Map([(keyed_module.Point(1.0, -0.0), "p")])  # a generated value object as a key
assert labels[keyed_module.Point(1.0, -0.0)] == "p" and keyed_module.Point(1.0, 0.0) not in labels